# Pandas Test Question

In [62]:
import numpy as np
import pandas as pd

get data

In [63]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

titanic_raw = pd.read_csv(url)


Write pandas code that:
1. Creates df as an independent copy.
2. Fills missing age values with the median age within each combination of sex and class.
3. Fills missing embarked values with the overall mode.
4. Converts class into an ordered categorical:
   Third < Second < First.
5. Creates age_group with bins:
   - [0, 18) → "child"
   - [18, 40) → "young_adult"
   - [40, 65) → "middle_age"
   - [65, ∞) → "older"
6. Returns one row per class, in the specified order, containing:
   - n_passengers
   - survival_rate
   - median_fare
   
Requirements:
- Do not use loops or apply(axis=1).
- Do not mutate the original titanic.
- Briefly explain why df["age"].fillna(...) without assignment does not ordinarily update df.

Grading: 10 points
- Copying and missing-age imputation: 3
- Mode imputation and ordered category: 2
- Correct half-open age bins: 2
- Correct grouped summary and ordering: 2
- Explanation of assignment behavior: 1
Reply with your code and explanation; optionally include elapsed time.

# Test Question

In [ ]:
df = titanic.copy()

# clean age -- are there missings?
df["age"].isna().sum()

# replace missings using fillna. Input into that a series with the same row indexes (calculated from df) that have the replacement values.
# It will swap over only when missing

# always put groupby on the df, then the column, then the transform
# don't forget to overwrite the original
df["age"] = df["age"].fillna(
    df.groupby(["sex", "class"])["age"].transform("median")
)

# embarked
df["embarked"].isna().sum()

df["embarked"] = df["embarked"].fillna(
    df["embarked"].mode().iloc[0]
)

df["embarked"].isna().sum()

df["class"] = pd.Categorical(
    df["class"],
    categories = ["Third", "Second", "First"],
    ordered = True
)

# cut bins -- could use pd.cut but i like np.select
df = (
    df
    .assign(
        age_group = lambda x: np.select(
            [
                x["age"] < 18,
                x["age"] < 40,
                x["age"] < 65,
                x["age"] >= 65

            ],
            [
                "child",
                "young_adult",
                "middle_age",
                "older"
            ],
            default = "undefined"

        )

    )
)

# grouped analysis
(
    df
    .groupby("class")
    .agg(
        n_passengers = ("survived", "size"),
        survival = ("survived", "mean"),
        median_fare = ("fare", "median")
    )
    .sort_values("class")
)


/var/folders/bj/45wmvwcj3njb1xt57ps1k86r0000gn/T/ipykernel_1214/715046093.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["sex", "class"])["age"].transform("median")


In [ ]:
df["class"] = pd.Categorical(
    df["class"],
    categories = ["Third", "Second", "First"],
    ordered = True
)

df = (
    df
    .assign(
        age_group = lambda x: np.select(
            [
                x["age"] < 18,
                x["age"] < 40,
                x["age"] < 65,
                x["age"] >= 65

            ],
            [
                "child",
                "young_adult",
                "middle_age",
                "older"
            ],
            default = "undefined"

        )

    )
)

(
    df
    .groupby("class")
    .agg(
        n_passengers = ("survived", "size"),
        survival = ("survived", "mean"),
        median_fare = ("fare", "median")
    )
    .sort_values("class")
)
